In [ ]:
# LightGBM 라이브러리를 불러온다.
# 현재 설치된 LightGBM의 버전을 확인하기 위해 사용한다.
import lightgbm

# 설치된 LightGBM 라이브러리의 버전을 출력한다.
print(lightgbm.__version__)

### LightGBM 적용 – 위스콘신 Breast Cancer Prediction

In [ ]:
# LightGBM의 파이썬 패키지인 lightgbm에서 LGBMClassifier 임포트
# LGBMClassifier는 LightGBM 기반의 분류 모델을 만들 때 사용한다.
from lightgbm import LGBMClassifier

# 데이터프레임 형태로 데이터를 다루기 위해 pandas를 불러온다.
import pandas as pd

# 수치 계산 및 배열 처리를 위해 numpy를 불러온다.
import numpy as np

# 사이킷런에서 제공하는 위스콘신 유방암 데이터셋을 불러오기 위해 사용한다.
from sklearn.datasets import load_breast_cancer

# 학습 데이터와 테스트 데이터를 나누기 위해 train_test_split을 불러온다.
from sklearn.model_selection import train_test_split

# 위스콘신 유방암 데이터셋을 불러온다.
dataset = load_breast_cancer()

# feature 데이터를 DataFrame 형태로 변환한다.
# dataset.data에는 입력 변수들이 들어 있고,
# dataset.feature_names에는 각 feature의 이름이 들어 있다.
cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)

# target 컬럼을 추가한다.
# target은 예측해야 할 정답 레이블이다.
cancer_df['target']= dataset.target

# cancer_df에서 feature 데이터와 label 데이터를 분리한다.
# 마지막 컬럼인 target을 제외한 나머지 컬럼은 입력 변수로 사용한다.
X_features = cancer_df.iloc[:, :-1]

# 마지막 컬럼인 target은 정답 레이블로 사용한다.
y_label = cancer_df.iloc[:, -1]

# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
# random_state를 설정하여 실행할 때마다 동일한 데이터 분할이 이루어지도록 한다.
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

# 위에서 만든 X_train, y_train을 다시 쪼개서 90%는 학습과 10%는 검증용 데이터로 분리
# 검증 데이터는 모델 학습 중 성능 평가와 조기 중단에 사용된다.
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

# 앞서 XGBoost와 동일하게 n_estimators는 400 설정.
# n_estimators는 생성할 부스팅 트리의 개수를 의미한다.
# learning_rate는 각 트리가 전체 모델에 반영되는 비율이다.
lgbm_wrapper = LGBMClassifier(n_estimators=400, learning_rate=0.05)

# LightGBM도 XGBoost와 동일하게 조기 중단 수행 가능.
# evals에는 학습 과정에서 평가할 데이터셋을 지정한다.
# 여기서는 학습 데이터와 검증 데이터를 함께 평가 대상으로 설정한다.
evals = [(X_tr, y_tr), (X_val, y_val)]

# LightGBM 모델을 학습시킨다.
# early_stopping_rounds=50은 검증 성능이 50번 반복 동안 개선되지 않으면 학습을 중단한다는 의미이다.
# eval_metric="logloss"는 평가 지표로 로그 손실을 사용한다는 의미이다.
# eval_set=evals는 학습 과정에서 평가할 데이터셋 목록이다.
# verbose=True는 학습 과정의 평가 결과를 출력하도록 한다.
lgbm_wrapper.fit(X_tr, y_tr, early_stopping_rounds=50, eval_metric="logloss", eval_set=evals, verbose=True)

# 학습된 LightGBM 모델을 이용하여 테스트 데이터의 클래스를 예측한다.
preds = lgbm_wrapper.predict(X_test)

# 테스트 데이터가 클래스 1에 속할 예측 확률값을 추출한다.
# predict_proba 결과에서 두 번째 컬럼이 클래스 1에 대한 확률이다.
pred_proba = lgbm_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
# 오차 행렬과 정확도를 계산하기 위해 confusion_matrix, accuracy_score를 불러온다.
from sklearn.metrics import confusion_matrix, accuracy_score

# 정밀도와 재현율을 계산하기 위해 precision_score, recall_score를 불러온다.
from sklearn.metrics import precision_score, recall_score

# F1 score와 ROC-AUC를 계산하기 위해 f1_score, roc_auc_score를 불러온다.
from sklearn.metrics import f1_score, roc_auc_score

# 분류 모델의 평가 지표를 한 번에 출력하는 함수를 정의한다.
def get_clf_eval(y_test, pred=None, pred_proba=None):
    # 오차 행렬을 계산한다.
    # 실제값과 예측값을 비교하여 TN, FP, FN, TP를 확인할 수 있다.
    confusion = confusion_matrix( y_test, pred)

    # 정확도는 전체 데이터 중 올바르게 예측한 비율이다.
    accuracy = accuracy_score(y_test , pred)

    # 정밀도는 양성으로 예측한 것 중 실제 양성의 비율이다.
    precision = precision_score(y_test , pred)

    # 재현율은 실제 양성 중 모델이 양성으로 맞게 예측한 비율이다.
    recall = recall_score(y_test , pred)

    # F1 score는 정밀도와 재현율의 조화 평균이다.
    f1 = f1_score(y_test,pred)

    # ROC-AUC 추가
    # ROC-AUC는 모델이 양성과 음성을 얼마나 잘 구분하는지 나타내는 지표이다.
    roc_auc = roc_auc_score(y_test, pred_proba)

    # 오차 행렬을 출력한다.
    print('오차 행렬')
    print(confusion)

    # ROC-AUC print 추가
    # 정확도, 정밀도, 재현율, F1, AUC를 한 번에 출력한다.
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
# 테스트 데이터의 실제값, 예측값, 예측 확률값을 이용하여
# LightGBM 모델의 분류 성능을 평가한다.
get_clf_eval(y_test, preds, pred_proba)

In [ ]:
# plot_importance( )를 이용하여 feature 중요도 시각화
# LightGBM 모델에서 각 feature가 예측에 얼마나 기여했는지 확인하기 위해 사용한다.
from lightgbm import plot_importance

# 그래프 시각화를 위해 matplotlib.pyplot을 불러온다.
import matplotlib.pyplot as plt

# 주피터 노트북 내부에 그래프가 바로 출력되도록 설정한다.
%matplotlib inline

# 피처 중요도 그래프의 크기를 설정한다.
fig, ax = plt.subplots(figsize=(10, 12))

# 학습된 LightGBM 모델의 feature 중요도를 시각화한다.
# 중요도가 높은 feature일수록 모델 예측에 더 많이 사용되었다고 볼 수 있다.
plot_importance(lgbm_wrapper, ax=ax)

# 생성된 feature importance 그래프를 tif 파일로 저장한다.
plt.savefig('lightgbm_feature_importance.tif', format='tif', dpi=300, bbox_inches='tight')